# Prepare VITS fine-tune (manifest, config, helper scripts)

Purpose
-------

- Produce a small training manifest from `data/processed/metadata.csv` (JSONL);
- Generate a minimal Coqui TTS (VITS) smoke config under `configs/`;
- Write a Windows helper script to launch a tiny smoke training run;
- Provide a small validator for the manifest.

Prerequisites
-------------
- You already ran preprocessing and validation (notebooks 3 and 4) 

In [ ]:
from pathlib import Path
from statistics import mean
import json
import csv
import soundfile as sf
import yaml

project_root = Path(".")
processed_wav_dir = project_root / "data/processed/wavs"
metadata_csv = project_root / "data/processed/metadata.csv"
manifests_dir = project_root / "data/processed/manifests"
configs_dir = project_root / "configs"
scripts_dir = project_root / "scripts"
outputs_dir = project_root / "outputs/smoke_vits"

manifests_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
scripts_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

smoke_num_examples = 50
smoke_batch_size = 4
smoke_max_steps = 200
sample_rate = 22050

## Let's build a small JSONL manifest

- Read LJSpeech-style `metadata.csv` (filename|transcript|speaker);
- Keep only entries with a non-empty transcript and an existing processed WAV;
- Save up to `smoke_num_examples` records to `data/processed/manifests/smoke_manifest.jsonl`;
    Each JSON line: { "wav": "<abs_path>", "text": "<transcript>", "duration" : <seconds>, "speaker": "<id>" } 

In [ ]:
manifest_path = manifests_dir / "smoke_manifest.jsonl"
written = 0

if not metadata_csv.exists():
    raise FileNotFoundError(f"metadata.csv not found: {metadata_csv}")

with open(metadata_csv, "r", encoding="utf-8") as metadata_file, open(manifest_path, "w", encoding="utf-8") as manifest_file:
    for line in metadata_file:
        line = line.strip()
        if not line:
            continue

        parts = [part.strip() for part in line.split("|")]

        if len(parts) == 1:
            parts = [parts[0], "", ""]
        elif len(parts) == 2:
            parts.append("")

        wav_filename, transcript, speaker_id = parts[:3]

        if not transcript:
            continue

        wav_path = processed_wav_dir / wav_filename

        if not wav_path.exists():
            continue

        info = sf.info(str(wav_path))
        duration = round((info.frames or 0) / (info.samplerate or sample_rate), 3)
        record = {"wav": str(wav_path.resolve()), "text": transcript, "duration": duration, "speaker": speaker_id}
        manifest_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        written += 1
        if written >= smoke_num_examples:
            break

print(f"Wrote {written} records -> {manifest_path}")

## Minimal Coqui TTS-style config for smoke test

- Small batch size, few steps, dataset points to the manifest above;
- Inspect and adapt learning rate, optimizer, and model params before running real training.

In [ ]:
config_path = configs_dir / "smoke_vits.yaml"

config = {
    "name": "vits_smoke",
    "output_path": str(outputs_dir.as_posix()),
    "datasets": [
        {
            "name": "smoke_dataset",
            "type": "jsonl",
            "manifest_path": str(manifest_path.as_posix()),
            "audio_sample_rate": sample_rate
        }
    ],
    "training": {
        "max_steps": smoke_max_steps,
        "batch_size": smoke_batch_size,
        "save_checkpoint_steps": 50,
        "log_steps": 10,
        "num_workers": 0
    },
    "model": {
        "name": "vits",
        "sample_rate": sample_rate,
        "n_mels": 80
    },
    "optimizer": {
        "learning_rate": 1e-4
    },
    "eval": {
        "sentences": ["Olá, este é um teste rápido.", "Bom dia, como vai?"]
    }
}

with open(config_path, "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False, allow_unicode=True)

print("Wrote config ->", config_path)

# Writing a Windows helper script to run a smoke training CLI

- This script activates the local venv and launches the Coqui TTS trainer with the new config;
- Edit the command if your TTS installation uses a different CLI.

In [ ]:
run_script_path = scripts_dir / "run_finetune_smoke.bat"
python_exe = project_root / ".venv" / "Scripts" / "python.exe"

bat_lines = [
    "@echo off",
    "REM Activate venv and run small VITS smoke training",
    f"call \"{(project_root / '.venv' / 'Scripts' / 'activate.bat').as_posix()}\"",
    f"\"{python_exe.as_posix()}\" -m TTS.bin.train --config_path \"{config_path.as_posix()}\" --continue_path \"\"",
    "pause",
]

with open(run_script_path, "w", encoding="utf-8", newline="\n") as file:
    file.write("\n".join(bat_lines))

print("Wrote run script ->", run_script_path)

## Validate the generated manifest

- Check each JSONL entry exists and durations are reasonable;
- This cell prints a short report, fix any missing files before training.

In [ ]:
errors = []
durations = []

with open(manifest_path, "r", encoding="utf-8") as file:
    for line in file:
        rec = json.loads(line)
        wav_file = Path(rec["wav"])
        if not wav_file.exists():
            errors.append(f"MISSING: {wav_file}")
            continue
        durations.append(rec.get("duration", 0.0))

total_records = sum(1 for _ in open(manifest_path, "r", encoding="utf-8"))
print("Records:", total_records)
print("Missing files:", len(errors))

for error in errors[:20]:
    print(" ", error)

if durations:
    print("Duration stats: count=", len(durations), "mean=", round(mean(durations), 3), "min=", min(durations), "max=", max(durations))
else:
    print("No durations found (manifest may be empty).")